# Week 1, Day 2: One question, three endpoints

**Lesson:** [week1/day2.ipynb](../../../week1/day2.ipynb), *The Chat Completions API, endpoints and the openai package*. **About 40 minutes.**

**What you practice here**

- Calling the Chat Completions endpoint by hand: a POST with your key as a Bearer token and a JSON body, and reading the answer out of the JSON reply.
- Making clients with `OpenAI` and a base URL and key for OpenAI's own endpoint, Gemini and Ollama, so the same Chat Completions call reaches each of them.
- The lesson's homework: the Day 1 summarizer, running on a local Llama model through Ollama.

**How this notebook works.** Run the first code cell once. Then, for each part below, write it in its cell, run that cell, and run the check under it. The checks hand your code stand-ins shaped like the real client and libraries: they record what your code sends and answer with canned replies, so nothing is called and nothing is spent. Write from memory, and look back at the lesson only when you are stuck.

When every check passes, save the notebook, run the checker and push, as in every session. The tracker then marks the lesson done.

In [ ]:
# Run this cell first. It finds plan/check.py in your fork and loads this exercise's checks.
import pathlib, sys
ROOT = next((p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "plan" / "check.py").is_file()), None)
if ROOT:
    sys.path.insert(0, str(ROOT / "plan"))
    try:
        from check import exercise
    except ImportError:
        raise ImportError("your plan/check.py is older than this notebook: run git pull, or copy the new one from the tracker page") from None
    check = exercise("week1/day2", globals())
else:  # in Colab, or outside your fork: write and try the parts here, and run the checks in your codespace
    def check(part=None):
        print("The checks are off here, as plan/check.py is not in a folder above this notebook. Run them in your codespace.")
    print("plan/check.py was not found above this notebook, so the checks are off here, as in Colab.")

## The task

The lesson calls a model in three ways: with a plain HTTP POST to `https://api.openai.com/v1/chat/completions`;
with the `openai` package, which only wraps that same POST; and with the same package pointed at another provider
through a new base URL. Its homework runs the Day 1 summarizer on a local model. Build all three here, on wrestling
history: a fun fact asked by hand, a client for each provider, and a summarizer for a page about WrestleMania III.

The two functions that make a call take what they call as their first parameter: `ask_over_http` takes the
`requests` module, and `summarize` takes a client. The checks hand them stand-ins with the same methods. These
record what you send and answer with a canned reply, so nothing goes over the network.

## What each call should give

```python
ask_over_http(requests, api_key, "Tell me a fun fact about WrestleMania")
# makes one POST to https://api.openai.com/v1/chat/completions, with your key as a Bearer token and this JSON body:
#   {"model": "gpt-5-nano", "messages": [{"role": "user", "content": "Tell me a fun fact about WrestleMania"}]}
# and returns the answer's text, such as 'WrestleMania III drew 93,173 fans to the Pontiac Silverdome.'

client_for("gemini", google_api_key).base_url  # URL('https://generativelanguage.googleapis.com/v1beta/openai/')
client_for("ollama").api_key                   # 'ollama'

summarize(client_for("ollama"), page)
# makes one request with the model llama3.2: your system message, then a user message holding the page,
# and returns the summary's text
```

In [ ]:
# The imports for this exercise. Add more here if you need them: the checks have the standard library, openai, pydantic, dotenv, numpy, bs4 and tiktoken.
from openai import OpenAI

### 1 · `ask_over_http(http, api_key, question, model="gpt-5-nano")`

Asks one question over plain HTTP, as the lesson's `requests.post` cell does, and returns the answer's text. `http` is
the `requests` module; the checks pass a stand-in with the same `post`. Send one POST to
`https://api.openai.com/v1/chat/completions` with two headers: an `Authorization` header holding `Bearer ` and your
key, and a `Content-Type` of `application/json`. Its body holds the `"model"` and a `"messages"` list whose only
message is a user message holding `question`; send it as JSON, as the lesson does. Return the first choice's message
content, read from the reply's JSON with dict keys: the SDK's objects, with their attributes, come only from the
`openai` package.

<details><summary>Hint</summary>

```python
headers = {"Authorization": f"Bearer {api_key}", "Content-Type": "application/json"}
payload = {"model": model, "messages": [{"role": "user", "content": question}]}
response = http.post("https://api.openai.com/v1/chat/completions", headers=headers, json=payload)
return response.json()["choices"][0]["message"]["content"]
```

</details>

In [ ]:
def ask_over_http(http, api_key, question, model="gpt-5-nano"):
    # Write ask_over_http here, as described above.
    raise NotImplementedError("ask_over_http is not written yet")

In [ ]:
check("ask_over_http")

### 2 · `client_for(provider, api_key=None)`

Builds an `OpenAI` client for one provider, without calling it. For `"openai"`, return a client for OpenAI's own
endpoint, with your key. For `"gemini"`, point the client at Gemini's OpenAI-compatible endpoint,
`https://generativelanguage.googleapis.com/v1beta/openai/`, with your key, as the lesson's Gemini cell does. For
`"ollama"`, point it at Ollama on your machine, `http://localhost:11434/v1`, with the key `"ollama"`, whatever
`api_key` you are given, as the lesson's Ollama cell does. Making a client sends nothing, so the checks use the real
`OpenAI` here and read its base URL and key. They try only these three providers.

<details><summary>Hint</summary>

```python
OpenAI(api_key=api_key)                                         # OpenAI's own endpoint
OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")  # another endpoint: its base_url and its key
```

</details>

In [ ]:
def client_for(provider, api_key=None):
    # Write client_for here, as described above.
    raise NotImplementedError("client_for is not written yet")

In [ ]:
check("client_for")

### 3 · `summarize(client, page, model="llama3.2")`

Summarizes a web page's text in one call, on a local Llama model by default. This is the homework: the Day 1
summarizer, moved to a model running on your machine. `page` is the page's text, as Day 1's `fetch_website_contents`
(in `week1/scraper.py`) returns it, so nothing is fetched here. Make one Chat Completions call with the model
`model` and two messages: a system message of your own (20 characters or more), then a user message holding an
instruction and the whole page. Return the reply's text.

<details><summary>Hint</summary>

```python
response = client.chat.completions.create(model=model, messages=[...])  # your system and user messages
return response.choices[0].message.content
```

</details>

In [ ]:
def summarize(client, page, model="llama3.2"):
    # Write summarize here, as described above.
    raise NotImplementedError("summarize is not written yet")

In [ ]:
check("summarize")

## Check everything

The cell below runs all 8 checks again. When they all pass, save this notebook (Ctrl+S, or Cmd+S on a Mac), then run the checker and push, as in every session:

```bash
uv run python plan/check.py
git add -A
git commit -m "Week 1, Day 2 exercise"
git push
```

The checker runs the same checks on the saved notebook, and the tracker marks the lesson done. It reads only your imports, functions, classes and constants, so the experiments in your other cells never get in the way.

In [ ]:
check()

## Try it for real

Now run what you wrote on the real thing. Write the steps in the cell at the end; they may call a model with your key, and the checks leave them out.

The GPT calls cost a fraction of a cent. The local steps need Ollama running (`ollama serve`) with `llama3.2` pulled
(`ollama pull llama3.2`). Gemini is optional: it needs `GOOGLE_API_KEY` in your `.env`.

1. Load your keys with `load_dotenv(override=True)`: set `openai_key` from `OPENAI_API_KEY`, and `google_key` from
   `GOOGLE_API_KEY` (or `""` if you have none).
2. Import `requests` and print `ask_over_http(requests, openai_key, "Tell me a fun fact about WrestleMania")`.
3. Ask the same question through the SDK on each provider you have: make the client with `client_for(...)`, then a
   Chat Completions call with `gpt-5-nano`, `gemini-3.1-flash-lite` or `llama3.2`. As the lesson does, skip Gemini
   when `google_key` does not start with `AIz` or `AQ.`.
4. The homework. Day 1's `fetch_website_contents` is in `week1/scraper.py`, so put that folder on `sys.path` first
   (`sys.path.insert(0, str(ROOT / "week1"))`), then import it from `scraper`. Set `page` to the text of the mobile
   page `https://en.m.wikipedia.org/wiki/WrestleMania_III` (desktop pages start with menus), and print its first 500
   characters: the start of what the model will get.
5. Print `summarize(client_for("ollama"), page)`. Then pull `deepseek-r1:1.5b` and try it with
   `model="deepseek-r1:1.5b"`.

<details><summary>Stuck? One way to write it</summary>

```python
import os
import sys

import requests
from dotenv import load_dotenv

load_dotenv(override=True)
openai_key = os.getenv("OPENAI_API_KEY")
google_key = os.getenv("GOOGLE_API_KEY") or ""

question = "Tell me a fun fact about WrestleMania"
print(ask_over_http(requests, openai_key, question))

for provider, key, model in [("openai", openai_key, "gpt-5-nano"),
                             ("gemini", google_key, "gemini-3.1-flash-lite"),
                             ("ollama", None, "llama3.2")]:
    if provider == "gemini" and not google_key.startswith(("AIz", "AQ.")):
        continue  # no Gemini key, so skip it, as the lesson does
    response = client_for(provider, key).chat.completions.create(
        model=model, messages=[{"role": "user", "content": question}])
    print(f"{provider}: {response.choices[0].message.content}\n")

sys.path.insert(0, str(ROOT / "week1"))  # Day 1's scraper.py
from scraper import fetch_website_contents

page = fetch_website_contents("https://en.m.wikipedia.org/wiki/WrestleMania_III")
print(page[:500])  # the start of what the model gets

print(summarize(client_for("ollama"), page))
print(summarize(client_for("ollama"), page, model="deepseek-r1:1.5b"))
```

</details>

In [ ]:
# Try it for real: write the steps above here, in as many cells as you like. This cell may call a model with your key; the checks leave it out.